In [6]:
# !pip3 install firebase-admin

In [7]:
import firebase_admin
from firebase_admin import credentials, firestore

In [8]:
# Path to your downloaded Firebase private key JSON file
cred = credentials.Certificate("./health-app-7a8b0-firebase-adminsdk-fbsvc-9d28c0ae3f.json")

# Initialize Firebase App (only once per script)
firebase_admin.initialize_app(cred)

# Initialize Firestore client
db = firestore.client()

# Get all top-level collections
collections = db.collections()

print("Collections in Firestore:")
for collection in collections:
    print(collection.id)

Collections in Firestore:
admins
chatHistory
doctors
emergencyContacts
insurance
medicalReports
medicines
personalContacts
personalDetails
pharmacies
userPreferences
users


In [39]:
def get_user_medical_history_and_medicine_summary(uid: str):
    # ---------------------------
    # 1. Get medical history
    # ---------------------------
    user_ref = db.collection("users").document(uid)
    doc = user_ref.get()

    if not doc.exists:
        return {"error": "User not found"}

    data = doc.to_dict()

    summaries = []
    if "answers" in data:
        for item in data["answers"]:
            sum_ans = item.get("summarizedAnswer")
            if sum_ans:
                summaries.append(sum_ans)

    # ---------------------------
    # 2. Get medicine information
    # ---------------------------
    meds_ref = db.collection("medicines")
    meds_query = meds_ref.where("userId", "==", uid).stream()

    medicines = []
    for med_doc in meds_query:
        med = med_doc.to_dict()
        medicines.append({
            "name": med.get("name"),
            "dosage": med.get("dosage"),
            "frequency": med.get("frequency")
        })

    # ---------------------------
    # 3. Build final response
    # ---------------------------
    return {
        "user_id": uid,
        "medical_history": summaries,
        "medicines": medicines
    }


# Call function
result = get_user_medical_history_and_medicine_summary("pGSZNP7t8odnxmOIpRbDcuthbbM2")

print(result)

# print(result["user_id"])
# print("\n")
# print(result["medical_history"])
# print("\n")
# print(result["medicines"])

{'user_id': 'pGSZNP7t8odnxmOIpRbDcuthbbM2', 'medical_history': ["Patient's date of birth is 07/06/1946.", 'Patient identifies as Male.', "Patient's ethnicity is Caucasian .", "Patient's address is 21377 Gosier Way.", 'Patient resides in Boca Raton.', 'Patient resides in Florida .', "Patient's zip code is 33428.", "Patient's phone number is 5617025533.", 'Patient\'s height is 5\'10".', "Patient's weight is 175 lbs.", 'Patient has had surgeries in the past.', 'Laser on left vocal cord ', 'Patient has been hospitalized in the past.', '2012 for vocal cord', 'Patient has high blood pressure.', 'Patient provided details about blood pressure.', 'Patient has diabetes.', 'Patient provided details about diabetes.', 'Patient does not have heart disease.', 'Patient does not have any allergies.', 'Patient currently smokes tobacco.', '1/2 pack a day', 'Patient consumes alcohol.', 'Socially ', 'Patient does not use recreational drugs.', 'Patient has not experienced recent weight changes.', 'Patient d